In [1]:
# ============================================================
# NOTEBOOK 05 · Evaluation & Validation
# Cell 1 · Setup and Data Loading
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
root     = Path("..")
data_dir = root / "data"

# ── Required input files ─────────────────────────────────────
REQUIRED_FILES = {
    # From 04 — metric-level scores (primary evaluation input)
    "metric_scores"      : data_dir / "metric_window_scores.csv",

    # From 04 — combined 03c vs 04 comparison table
    "comparison"         : data_dir / "combined_vs_03c_comparison.csv",

    # From 03c — aggregate window-level scores
    "window_scores"      : data_dir / "sensor_window_scores.csv",

    # From 03b — neighbourhood metadata (isolate flag, neighbour count)
    "neighbour_meta"     : data_dir / "sensor_neighbour_metadata.csv",

    # From 03b — hourly features (for case-study time-series)
    "hour_features"      : data_dir / "sensor_hour_features.csv",

    # From 03b — metric list
    "model_metrics"      : data_dir / "model_metric_list.csv",
}

# ── Validate all files exist before loading ───────────────────
missing = [name for name, path in REQUIRED_FILES.items() if not path.exists()]
if missing:
    print("=" * 60)
    print("MISSING INPUT FILES — run earlier notebooks first:")
    for name in missing:
        print(f"  ✗  {name}  →  {REQUIRED_FILES[name]}")
    print("=" * 60)
    raise FileNotFoundError(f"{len(missing)} required file(s) missing.")

print("All required files found.\n")

# ── Load data ─────────────────────────────────────────────────
metric_scores  = pd.read_csv(REQUIRED_FILES["metric_scores"])
comparison     = pd.read_csv(REQUIRED_FILES["comparison"])
window_scores  = pd.read_csv(REQUIRED_FILES["window_scores"])
neighbour_meta = pd.read_csv(REQUIRED_FILES["neighbour_meta"])
hour_features  = pd.read_csv(REQUIRED_FILES["hour_features"])
model_metrics  = pd.read_csv(REQUIRED_FILES["model_metrics"])["metric"].tolist()

# ── Parse timestamps ──────────────────────────────────────────
metric_scores["window_start"]  = pd.to_datetime(metric_scores["window_start"])
window_scores["window_start"]  = pd.to_datetime(window_scores["window_start"])
comparison["window_start"]     = pd.to_datetime(comparison["window_start"])
hour_features["timestamp_utc"] = pd.to_datetime(hour_features["timestamp_utc"])

# ── Display settings ──────────────────────────────────────────
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows",    50)

# ── Summary ───────────────────────────────────────────────────
print("=" * 60)
print("DATA LOADED — SUMMARY")
print("=" * 60)
print(f"  metric_window_scores    : {metric_scores.shape[0]:,} rows  ×  {metric_scores.shape[1]} cols")
print(f"  combined_vs_03c         : {comparison.shape[0]:,} rows  ×  {comparison.shape[1]} cols")
print(f"  sensor_window_scores    : {window_scores.shape[0]:,} rows  ×  {window_scores.shape[1]} cols")
print(f"  sensor_neighbour_meta   : {neighbour_meta.shape[0]:,} rows  ×  {neighbour_meta.shape[1]} cols")
print(f"  sensor_hour_features    : {hour_features.shape[0]:,} rows  ×  {hour_features.shape[1]} cols")
print(f"  model_metrics           : {len(model_metrics)} metrics")
print("-" * 60)
print(f"  Unique sensors          : {metric_scores['ateccid'].nunique()}")
print(f"  Date range              : {metric_scores['window_start'].min().date()}  →  {metric_scores['window_start'].max().date()}")
print(f"  Metrics tracked         : {model_metrics}")
print("=" * 60)

All required files found.

DATA LOADED — SUMMARY
  metric_window_scores    : 14,784 rows  ×  14 cols
  combined_vs_03c         : 1,056 rows  ×  21 cols
  sensor_window_scores    : 1,056 rows  ×  23 cols
  sensor_neighbour_meta   : 33 rows  ×  3 cols
  sensor_hour_features    : 25,344 rows  ×  573 cols
  model_metrics           : 14 metrics
------------------------------------------------------------
  Unique sensors          : 33
  Date range              : 2025-08-15  →  2025-09-15
  Metrics tracked         : ['tvoc', 'co2', 'temp', 'pressure', 'lux', 'snd', 'humid', 'aqi1', 'aqi2', 'blue_relative', 'clear_relative', 'green_relative', 'red_relative', 'voc_resistance']


In [3]:
# ============================================================
# Cell 2 · Column Discovery (diagnostic — run once)
# ============================================================

for name, df in [
    ("metric_window_scores", metric_scores),
    ("sensor_window_scores", window_scores),
    ("combined_vs_03c",      comparison),
    ("sensor_neighbour_meta",neighbour_meta),
]:
    print(f"\n{'=' * 60}")
    print(f"  {name}  —  shape: {df.shape}")
    print(f"{'=' * 60}")
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
        print(f"  {col:<45} sample: {sample}")


  metric_window_scores  —  shape: (14784, 14)
  ateccid                                       sample: 0123067D065E709F01
  window_start                                  sample: 2025-08-15 00:00:00+00:00
  window_end                                    sample: 2025-08-16 00:00:00+00:00
  metric                                        sample: tvoc
  metric_basis                                  sample: self_history_only
  metric_rule_score                             sample: 0.0
  metric_model_residual_score                   sample: 0.5
  metric_driftscore_final                       sample: 0.075
  metric_confidence                             sample: 0.48
  metric_faultflag_final                        sample: 0
  metric_faulttype_rule                         sample: normal
  neighbour_count                               sample: 0
  mean_neigh_coverage                           sample: 0.0
  hours_scored                                  sample: 0

  sensor_window_scores  —  shape: (105

In [4]:
# ============================================================
# Cell 2 · Schema Validation & Data Quality Checks
# ============================================================

# ── Standardise sensor ID column name ────────────────────────
# sensor_window_scores uses 'sensor_id'; all other tables use 'ateccid'
# Rename once here so all downstream cells use 'ateccid' consistently
if "sensor_id" in window_scores.columns and "ateccid" not in window_scores.columns:
    window_scores = window_scores.rename(columns={"sensor_id": "ateccid"})
    print("  ✓  Renamed 'sensor_id' → 'ateccid' in sensor_window_scores")

# ── Derive isolated sensor flag from neighbour_count ─────────
# No explicit is_isolated column exists; neighbour_count == 0 is equivalent
neighbour_meta["is_isolated"] = neighbour_meta["neighbour_count"] == 0
n_isolated = neighbour_meta["is_isolated"].sum()
print(f"  ✓  Derived is_isolated flag  ({n_isolated} isolated sensors)\n")

# ── Schema checks ─────────────────────────────────────────────
def check_schema(df, name, required_cols):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  ✗  {name} — missing columns: {missing}")
        return False
    print(f"  ✓  {name} — all required columns present")
    return True

print("=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)

schemas_ok = True
schemas_ok &= check_schema(metric_scores, "metric_window_scores", [
    "ateccid", "window_start", "metric",
    "metric_driftscore_final", "metric_faulttype_rule",
    "metric_faultflag_final", "metric_confidence",
    "neighbour_count"
])
schemas_ok &= check_schema(window_scores, "sensor_window_scores", [
    "ateccid", "window_start", "drift_score", "binary_fault_flag",
    "overall_confidence"
])
schemas_ok &= check_schema(comparison, "combined_vs_03c", [
    "ateccid", "window_start",
    "combined_drift_score", "combined_binary_fault_flag",
    "combined_drift_score_03c", "combined_binary_fault_flag_03c",
    "drift_score_diff", "flag_agreement"
])
schemas_ok &= check_schema(neighbour_meta, "sensor_neighbour_meta", [
    "ateccid", "neighbour_count", "is_isolated"
])

if not schemas_ok:
    raise ValueError("Schema validation failed — fix missing columns before continuing.")

# ── Duplicate checks ──────────────────────────────────────────
print("\n" + "=" * 60)
print("DUPLICATE CHECKS")
print("=" * 60)

dup_metric = metric_scores.duplicated(subset=["ateccid", "window_start", "metric"]).sum()
dup_window = window_scores.duplicated(subset=["ateccid", "window_start"]).sum()
dup_comp   = comparison.duplicated(subset=["ateccid", "window_start"]).sum()

for label, count in [
    ("metric_window_scores  (ateccid, window_start, metric)", dup_metric),
    ("sensor_window_scores  (ateccid, window_start)",         dup_window),
    ("combined_vs_03c       (ateccid, window_start)",         dup_comp),
]:
    flag = "⚠️ " if count > 0 else "✓ "
    print(f"  {flag}  {label}  →  {count} duplicates")

# ── Null checks ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("NULL CHECKS — KEY COLUMNS")
print("=" * 60)

null_checks = {
    "metric_scores  · metric_driftscore_final"  : metric_scores["metric_driftscore_final"].isna().sum(),
    "metric_scores  · metric_faulttype_rule"    : metric_scores["metric_faulttype_rule"].isna().sum(),
    "metric_scores  · metric_faultflag_final"   : metric_scores["metric_faultflag_final"].isna().sum(),
    "window_scores  · drift_score"              : window_scores["drift_score"].isna().sum(),
    "window_scores  · binary_fault_flag"        : window_scores["binary_fault_flag"].isna().sum(),
}

for label, count in null_checks.items():
    flag = "⚠️ " if count > 0 else "✓ "
    print(f"  {flag}  {label:<50}  nulls: {count}")

# ── Fault type inventory ──────────────────────────────────────
print("\n" + "=" * 60)
print("FAULT TYPE INVENTORY  (metric_window_scores)")
print("=" * 60)
print(metric_scores["metric_faulttype_rule"].value_counts().to_string())

# ── Neighbour metadata summary ────────────────────────────────
print("\n" + "=" * 60)
print("NEIGHBOUR METADATA SUMMARY")
print("=" * 60)
print(f"  Total sensors     : {len(neighbour_meta)}")
print(f"  Isolated (n=0)    : {n_isolated}")
print(f"  With neighbours   : {len(neighbour_meta) - n_isolated}")
print(f"\n  neighbour_count distribution:")
print(neighbour_meta["neighbour_count"].value_counts().sort_index().to_string())

# ── Sensor coverage ───────────────────────────────────────────
print("\n" + "=" * 60)
print("SENSOR COVERAGE CHECK")
print("=" * 60)
windows_per_sensor = metric_scores.groupby("ateccid")["window_start"].nunique()
print(f"  Windows per sensor — min: {windows_per_sensor.min()}  "
      f"max: {windows_per_sensor.max()}  "
      f"mean: {windows_per_sensor.mean():.1f}")
if windows_per_sensor.min() < windows_per_sensor.max():
    print("  ⚠️  Unequal coverage — sensors with fewer windows:")
    print(windows_per_sensor[windows_per_sensor < windows_per_sensor.max()].to_string())
else:
    print("  ✓  All sensors have equal window coverage.")

print("\nCell 2 complete — ready for Cell 3.")

  ✓  Renamed 'sensor_id' → 'ateccid' in sensor_window_scores
  ✓  Derived is_isolated flag  (10 isolated sensors)

SCHEMA VALIDATION
  ✓  metric_window_scores — all required columns present
  ✓  sensor_window_scores — all required columns present
  ✓  combined_vs_03c — all required columns present
  ✓  sensor_neighbour_meta — all required columns present

DUPLICATE CHECKS
  ✓   metric_window_scores  (ateccid, window_start, metric)  →  0 duplicates
  ✓   sensor_window_scores  (ateccid, window_start)  →  0 duplicates
  ✓   combined_vs_03c       (ateccid, window_start)  →  0 duplicates

NULL CHECKS — KEY COLUMNS
  ✓   metric_scores  · metric_driftscore_final            nulls: 0
  ✓   metric_scores  · metric_faulttype_rule              nulls: 0
  ✓   metric_scores  · metric_faultflag_final             nulls: 0
  ✓   window_scores  · drift_score                        nulls: 0
  ✓   window_scores  · binary_fault_flag                  nulls: 0

FAULT TYPE INVENTORY  (metric_window_scores)
me

In [5]:
# ============================================================
# Cell 3 · Spatial Coherence Analysis
# 
# Research question answered:
#   When a sensor flags a fault, do its neighbours agree?
#   High agreement → genuine environmental event (not a fault)
#   Low agreement  → sensor-specific fault
# ============================================================

# ── Parse neighbour lists from neighbour_meta ─────────────────
neighbour_lookup = {}
for _, row in neighbour_meta.iterrows():
    sensor = row["ateccid"]
    if pd.isna(row["neighbours"]) or row["neighbours"] == "":
        neighbour_lookup[sensor] = []
    else:
        neighbour_lookup[sensor] = [n.strip() for n in str(row["neighbours"]).split(",")]

print("Neighbour lookup built.")
print(f"  Sensors with neighbours  : {sum(1 for v in neighbour_lookup.values() if len(v) > 0)}")
print(f"  Isolated sensors (empty) : {sum(1 for v in neighbour_lookup.values() if len(v) == 0)}")

# ── Build fault index for fast lookup ────────────────────────
# {(ateccid, window_start_str, metric) → 1/0}
metric_scores["window_start_str"] = metric_scores["window_start"].astype(str)
fault_index = metric_scores.set_index(
    ["ateccid", "window_start_str", "metric"]
)["metric_faultflag_final"].to_dict()

print("\nFault index built.")

# ── Compute spatial coherence per fault window ────────────────
fault_windows = metric_scores[metric_scores["metric_faultflag_final"] == 1].copy()
fault_windows["window_start_str"] = fault_windows["window_start"].astype(str)

results = []
for _, row in fault_windows.iterrows():
    sensor  = row["ateccid"]
    window  = row["window_start_str"]
    metric  = row["metric"]
    neighbours = neighbour_lookup.get(sensor, [])

    if len(neighbours) == 0:
        # Isolated — cannot evaluate spatially
        results.append({
            "ateccid"              : sensor,
            "window_start"         : row["window_start"],
            "metric"               : metric,
            "fault_type"           : row["metric_faulttype_rule"],
            "n_neighbours"         : 0,
            "n_neighbours_faulted" : np.nan,
            "agreement_rate"       : np.nan,
            "spatial_class"        : "isolated_sensor",
        })
        continue

    n_faulted = sum(
        fault_index.get((nb, window, metric), 0)
        for nb in neighbours
    )
    agreement_rate = n_faulted / len(neighbours)

    if agreement_rate >= 0.5:
        spatial_class = "shared_event"       # likely environmental
    elif agreement_rate > 0:
        spatial_class = "partial_agreement"  # mixed signal
    else:
        spatial_class = "isolated_fault"     # likely sensor-specific

    results.append({
        "ateccid"              : sensor,
        "window_start"         : row["window_start"],
        "metric"               : metric,
        "fault_type"           : row["metric_faulttype_rule"],
        "n_neighbours"         : len(neighbours),
        "n_neighbours_faulted" : n_faulted,
        "agreement_rate"       : agreement_rate,
        "spatial_class"        : spatial_class,
    })

coherence_df = pd.DataFrame(results)
print(f"\nSpatial coherence computed for {len(coherence_df):,} fault windows.")

# ── Overall spatial class distribution ───────────────────────
print("\n" + "=" * 60)
print("SPATIAL CLASS DISTRIBUTION  (all fault windows)")
print("=" * 60)
class_counts = coherence_df["spatial_class"].value_counts()
class_pct    = (class_counts / len(coherence_df) * 100).round(1)
for cls in class_counts.index:
    print(f"  {cls:<25}  {class_counts[cls]:>5}   ({class_pct[cls]}%)")

# ── Breakdown by fault type ───────────────────────────────────
print("\n" + "=" * 60)
print("SPATIAL CLASS  ×  FAULT TYPE")
print("=" * 60)
cross = pd.crosstab(
    coherence_df["fault_type"],
    coherence_df["spatial_class"],
    margins=True,
    margins_name="TOTAL"
)
print(cross.to_string())

# ── Agreement rate by metric ──────────────────────────────────
print("\n" + "=" * 60)
print("MEAN NEIGHBOUR AGREEMENT RATE BY METRIC")
print("(non-isolated fault windows only)")
print("=" * 60)
metric_agreement = (
    coherence_df[coherence_df["spatial_class"] != "isolated_sensor"]
    .groupby("metric")["agreement_rate"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
    .round(3)
)
metric_agreement.columns = ["mean_agreement", "median_agreement", "n_fault_windows"]
print(metric_agreement.to_string())

# ── Per-sensor isolated fault rate ────────────────────────────
print("\n" + "=" * 60)
print("SENSORS WITH HIGHEST ISOLATED FAULT RATE")
print("(only non-isolated sensors — higher rate = more suspicious)")
print("=" * 60)
sensor_class = (
    coherence_df[coherence_df["spatial_class"] != "isolated_sensor"]
    .groupby("ateccid")["spatial_class"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .round(3)
)
if "isolated_fault" in sensor_class.columns:
    sensor_class = sensor_class.sort_values("isolated_fault", ascending=False)
print(sensor_class.to_string())

# ── Save for later use ────────────────────────────────────────
coherence_df.to_csv(data_dir / "spatial_coherence.csv", index=False)
print("\nSaved → data/spatial_coherence.csv")
print("\nCell 3 complete — ready for Cell 4.")

Neighbour lookup built.
  Sensors with neighbours  : 23
  Isolated sensors (empty) : 10

Fault index built.

Spatial coherence computed for 3,591 fault windows.

SPATIAL CLASS DISTRIBUTION  (all fault windows)
  partial_agreement           1586   (44.2%)
  isolated_sensor             1532   (42.7%)
  isolated_fault               399   (11.1%)
  shared_event                  74   (2.1%)

SPATIAL CLASS  ×  FAULT TYPE
spatial_class         isolated_fault  isolated_sensor  partial_agreement  shared_event  TOTAL
fault_type                                                                                   
drift_abrupt                      95                0                352            49    496
drift_gradual                    288             1393               1147            25   2853
noisy_sensor                      16               88                 87             0    191
self_history_anomaly               0               51                  0             0     51
TOTAL            

In [7]:
# ============================================================
# Cell 4 · Temporal Persistence Analysis
#
# Research question answered:
#   Are fault labels stable and persistent across windows,
#   or are they one-off noise?
#
#   drift_gradual  → should show multi-window runs (persistent)
#   spike_transient → should show isolated single windows
#   drift_abrupt   → should show a step-change then persist
# ============================================================

# ── Sort data ─────────────────────────────────────────────────
metric_scores_sorted = metric_scores.sort_values(
    ["ateccid", "metric", "window_start"]
).copy()

# ── Compute consecutive fault run lengths ─────────────────────
# For each sensor-metric pair, count how many consecutive windows
# are flagged as faulted — a "fault run"

records = []
for (sensor, metric), grp in metric_scores_sorted.groupby(["ateccid", "metric"]):
    grp = grp.reset_index(drop=True)

    # Assign a run_id that increments each time the fault flag changes
    grp["run_id"] = (grp["metric_faultflag_final"] != grp["metric_faultflag_final"].shift()).cumsum()

    for run_id, run in grp.groupby("run_id"):
        is_fault   = run["metric_faultflag_final"].iloc[0] == 1
        run_length = len(run)
        fault_type = run["metric_faulttype_rule"].mode()[0]

        records.append({
            "ateccid"        : sensor,
            "metric"         : metric,
            "run_id"         : run_id,
            "is_fault_run"   : is_fault,
            "run_length"     : run_length,
            "fault_type"     : fault_type,
            "window_start"   : run["window_start"].iloc[0],
            "window_end"     : run["window_start"].iloc[-1],
        })

runs_df = pd.DataFrame(records)
fault_runs = runs_df[runs_df["is_fault_run"]].copy()

print(f"Total fault runs identified : {len(fault_runs):,}")
print(f"Total non-fault runs        : {(~runs_df['is_fault_run']).sum():,}")

# ── Run length distribution by fault type ─────────────────────
print("\n" + "=" * 60)
print("RUN LENGTH DISTRIBUTION BY FAULT TYPE")
print("(how many consecutive windows share the same fault label)")
print("=" * 60)

persistence = (
    fault_runs.groupby("fault_type")["run_length"]
    .agg(
        n_runs   ="count",
        mean     ="mean",
        median   ="median",
        max      ="max",
        pct_single = lambda x: (x == 1).mean() * 100,  # % one-off faults
        pct_multi  = lambda x: (x >= 3).mean() * 100,  # % runs of 3+ windows
    )
    .round(2)
    .sort_values("median", ascending=False)
)
persistence.columns = ["n_runs", "mean_windows", "median_windows",
                       "max_windows", "% single-window", "% 3+ windows"]
print(persistence.to_string())

# ── Key persistence check ──────────────────────────────────────
print("\n" + "=" * 60)
print("PERSISTENCE HYPOTHESIS CHECK")
print("=" * 60)

for ft in ["drift_gradual", "spike_transient", "drift_abrupt", "noisy_sensor"]:
    if ft not in persistence.index:
        continue
    row = persistence.loc[ft]
    print(f"\n  {ft}")
    print(f"    median run length : {row['median_windows']} windows")
    print(f"    % single-window   : {row['% single-window']}%")
    print(f"    % 3+ windows      : {row['% 3+ windows']}%")

# ── Per-sensor persistence — canteen sensors highlighted ───────
print("\n" + "=" * 60)
print("PER-SENSOR FAULT PERSISTENCE")
print("(mean consecutive fault windows per run, non-isolated sensors)")
print("=" * 60)

CANTEEN_IDS = ["0123EA796859295001", "0123690E6F25C93201", "01230B43B90B9BB401"]

sensor_persistence = (
    fault_runs
    .merge(neighbour_meta[["ateccid", "is_isolated"]], on="ateccid", how="left")
    .query("is_isolated == False")
    .groupby("ateccid")["run_length"]
    .agg(
        n_fault_runs = "count",
        mean_run_len = "mean",
        max_run_len  = "max",
        pct_persistent = lambda x: (x >= 3).mean() * 100
    )
    .round(2)
    .sort_values("mean_run_len", ascending=False)
)
sensor_persistence.columns = ["n_fault_runs", "mean_run_len", "max_run_len", "% persistent (3+)"]

# Mark canteen sensors
sensor_persistence["canteen"] = sensor_persistence.index.isin(CANTEEN_IDS)
print(sensor_persistence.to_string())

# ── Canteen sensor drill-down ──────────────────────────────────
print("\n" + "=" * 60)
print("CANTEEN SENSOR DRILL-DOWN (sensors 23, 27, 28)")
print("=" * 60)

canteen_runs = (
    fault_runs[fault_runs["ateccid"].isin(CANTEEN_IDS)]
    .sort_values(["ateccid", "window_start"])
    [["ateccid", "metric", "fault_type", "run_length", "window_start", "window_end"]]
)
print(canteen_runs.to_string(index=False))

# ── Temporal heatmap data — fault flag over time per sensor ───
print("\n" + "=" * 60)
print("DAILY FAULT FLAG RATE  (across all metrics, all sensors)")
print("(mean metric_faultflag_final per window date)")
print("=" * 60)
daily_fault_rate = (
    metric_scores
    .groupby("window_start")["metric_faultflag_final"]
    .mean()
    .round(3)
    .rename("mean_fault_rate")
)
print(daily_fault_rate.to_string())

# ── Save ──────────────────────────────────────────────────────
fault_runs.to_csv(data_dir / "fault_runs.csv", index=False)
print("\nSaved → data/fault_runs.csv")
print("\nCell 4 complete — ready for Cell 5.")

Total fault runs identified : 1,162
Total non-fault runs        : 1,450

RUN LENGTH DISTRIBUTION BY FAULT TYPE
(how many consecutive windows share the same fault label)
                      n_runs  mean_windows  median_windows  max_windows  % single-window  % 3+ windows
fault_type                                                                                            
self_history_anomaly      12          3.08             3.0            6            41.67         50.00
drift_abrupt             177          3.09             2.0           32            42.37         28.25
drift_gradual            938          3.11             2.0           32            41.90         42.00
noisy_sensor              35          2.49             1.0            6            57.14         34.29

PERSISTENCE HYPOTHESIS CHECK

  drift_gradual
    median run length : 2.0 windows
    % single-window   : 41.9%
    % 3+ windows      : 42.0%

  drift_abrupt
    median run length : 2.0 windows
    % single-windo

In [8]:
# ============================================================
# Cell 5 · Weekday/Weekend Fault Score Correction
#
# Problem:
#   The pipeline uses a single fleet-wide threshold to flag
#   faults regardless of day. But weekday fault rates (~0.30)
#   are nearly double weekend rates (~0.155), meaning occupancy
#   cycles are inflating false positives on weekdays.
#
# Approach:
#   1. Split windows into WEEKDAY (Mon-Fri) and WEEKEND (Sat-Sun)
#   2. For each regime, compute the fleet-wide percentile
#      distribution of metric_driftscore_final
#   3. Set a regime-specific threshold at the same PERCENTILE
#      as the original threshold — so the flagging logic is
#      equally strict within each regime, not across regimes
#   4. Re-flag, compare old vs new, and save corrected table
# ============================================================

# ── Step 1: Label each window by regime ───────────────────────
ms = metric_scores.copy()
ms["day_of_week"] = ms["window_start"].dt.dayofweek   # 0=Mon, 6=Sun
ms["regime"]      = ms["day_of_week"].apply(
    lambda d: "weekday" if d < 5 else "weekend"
)

print("=== WINDOW COUNT BY REGIME ===")
print(ms.groupby("regime")["window_start"].nunique().rename("unique_windows"))
print(f"\nTotal rows: {len(ms):,}")

# ── Step 2: Find what percentile the original threshold sits at
# We estimate the original threshold by finding the percentile
# at which ~26% of scores are flagged (the observed fleet average)
ORIGINAL_FLAG_RATE = ms["metric_faultflag_final"].mean()
ORIGINAL_THRESHOLD_PERCENTILE = (1 - ORIGINAL_FLAG_RATE) * 100

print(f"\n=== ORIGINAL THRESHOLD CALIBRATION ===")
print(f"  Original fleet-wide flag rate : {ORIGINAL_FLAG_RATE:.3f}")
print(f"  Implies threshold percentile  : {ORIGINAL_THRESHOLD_PERCENTILE:.1f}th percentile")

# Confirm: what score value is at that percentile?
original_thresh_value = np.percentile(
    ms["metric_driftscore_final"].dropna(), ORIGINAL_THRESHOLD_PERCENTILE
)
print(f"  Threshold score value         : {original_thresh_value:.4f}")

# Cross-check: flag rate if we apply this percentile-based threshold
cross_check = (ms["metric_driftscore_final"] >= original_thresh_value).mean()
print(f"  Cross-check flag rate         : {cross_check:.3f}  (should ≈ {ORIGINAL_FLAG_RATE:.3f})")

# ── Step 3: Compute regime-specific thresholds at SAME percentile
print(f"\n=== REGIME-SPECIFIC THRESHOLDS ({ORIGINAL_THRESHOLD_PERCENTILE:.1f}th percentile) ===")
regime_thresholds = {}
for regime, grp in ms.groupby("regime"):
    scores = grp["metric_driftscore_final"].dropna()
    thresh = np.percentile(scores, ORIGINAL_THRESHOLD_PERCENTILE)
    regime_thresholds[regime] = thresh
    print(f"  {regime:<10}  threshold = {thresh:.4f}  "
          f"(score distribution: mean={scores.mean():.3f}, "
          f"std={scores.std():.3f}, "
          f"p75={np.percentile(scores,75):.3f})")

# ── Step 4: Apply regime-specific thresholds ──────────────────
ms["corrected_threshold"] = ms["regime"].map(regime_thresholds)
ms["metric_faultflag_corrected"] = (
    ms["metric_driftscore_final"] >= ms["corrected_threshold"]
).astype(int)

# ── Step 5: Before vs after comparison ────────────────────────
print("\n=== BEFORE vs AFTER CORRECTION ===")
comparison = (
    ms.groupby("regime")
    .agg(
        original_flag_rate = ("metric_faultflag_final", "mean"),
        corrected_flag_rate= ("metric_faultflag_corrected", "mean"),
        n_rows             = ("metric_faultflag_final", "count"),
    )
    .round(3)
)
comparison["absolute_change"] = (
    comparison["corrected_flag_rate"] - comparison["original_flag_rate"]
).round(3)
print(comparison.to_string())

# ── Step 6: Which flags CHANGED? ──────────────────────────────
ms["flag_changed"]       = ms["metric_faultflag_final"] != ms["metric_faultflag_corrected"]
ms["newly_cleared"]      = (ms["metric_faultflag_final"] == 1) & (ms["metric_faultflag_corrected"] == 0)
ms["newly_flagged"]      = (ms["metric_faultflag_final"] == 0) & (ms["metric_faultflag_corrected"] == 1)

print("\n=== FLAG CHANGE SUMMARY ===")
print(f"  Total flags changed    : {ms['flag_changed'].sum():,}  "
      f"({ms['flag_changed'].mean()*100:.1f}% of all rows)")
print(f"  Newly CLEARED (FP→OK)  : {ms['newly_cleared'].sum():,}  "
      f"(weekday over-flagging removed)")
print(f"  Newly FLAGGED (OK→FP)  : {ms['newly_flagged'].sum():,}  "
      f"(weekend under-flagging corrected)")

# ── Step 7: Per-sensor impact — which sensors benefited most? ─
print("\n=== PER-SENSOR FLAG RATE: ORIGINAL vs CORRECTED ===")
sensor_compare = (
    ms.groupby("ateccid")
    .agg(
        original  = ("metric_faultflag_final", "mean"),
        corrected = ("metric_faultflag_corrected", "mean"),
    )
    .round(3)
)
sensor_compare["delta"] = (sensor_compare["corrected"] - sensor_compare["original"]).round(3)
sensor_compare = sensor_compare.sort_values("delta")

print("  Biggest REDUCTIONS (occupancy was inflating faults):")
print(sensor_compare.head(5).to_string())
print("\n  Biggest INCREASES (weekends were under-flagging real faults):")
print(sensor_compare.tail(5).to_string())

# ── Step 8: Day-of-week fault rate AFTER correction ───────────
print("\n=== FLEET-WIDE FAULT RATE BY DAY  — BEFORE vs AFTER ===")
dow_compare = (
    ms.groupby(["day_of_week", ms["window_start"].dt.day_name()])
    .agg(
        before = ("metric_faultflag_final", "mean"),
        after  = ("metric_faultflag_corrected", "mean"),
    )
    .round(3)
    .reset_index()
    .sort_values("day_of_week")
)
dow_compare.columns = ["day_num", "day_name", "before", "after"]
print(dow_compare[["day_name","before","after"]].to_string(index=False))

# ── Step 9: Per-metric impact ──────────────────────────────────
print("\n=== PER-METRIC FLAG RATE: ORIGINAL vs CORRECTED ===")
metric_compare = (
    ms.groupby("metric")
    .agg(
        original  = ("metric_faultflag_final", "mean"),
        corrected = ("metric_faultflag_corrected", "mean"),
    )
    .round(3)
)
metric_compare["delta"] = (metric_compare["corrected"] - metric_compare["original"]).round(3)
metric_compare = metric_compare.sort_values("delta")
print(metric_compare.to_string())

# ── Step 10: Canteen sensors — did correction help? ───────────
print("\n=== CANTEEN SENSOR CORRECTION IMPACT ===")
CANTEEN = {
    "0123EA796859295001": "S23",
    "0123690E6F25C93201": "S27",
    "01230B43B90B9BB401": "S28",
}
canteen_compare = (
    ms[ms["ateccid"].isin(CANTEEN.keys())]
    .groupby(["ateccid", "regime"])
    .agg(
        original  = ("metric_faultflag_final", "mean"),
        corrected = ("metric_faultflag_corrected", "mean"),
    )
    .round(3)
    .reset_index()
)
canteen_compare["label"] = canteen_compare["ateccid"].map(CANTEEN)
print(canteen_compare[["label","regime","original","corrected"]].to_string(index=False))

# ── Save corrected scores ──────────────────────────────────────
corrected_scores = ms[[
    "ateccid", "metric", "window_start", "regime",
    "metric_driftscore_final",
    "metric_faultflag_final",        # original
    "metric_faultflag_corrected",    # corrected
    "metric_faulttype_rule",
    "newly_cleared", "newly_flagged", "flag_changed",
]].copy()

corrected_scores.to_csv(data_dir / "metric_scores_corrected.csv", index=False)
print("\nSaved → data/metric_scores_corrected.csv")
print("\nCell 5 complete — ready for Cell 6 (visualisation) or Cell 7 (synthetic injection).")

=== WINDOW COUNT BY REGIME ===
regime
weekday    22
weekend    10
Name: unique_windows, dtype: int64

Total rows: 14,784

=== ORIGINAL THRESHOLD CALIBRATION ===
  Original fleet-wide flag rate : 0.243
  Implies threshold percentile  : 75.7th percentile
  Threshold score value         : 0.7002
  Cross-check flag rate         : 0.243  (should ≈ 0.243)

=== REGIME-SPECIFIC THRESHOLDS (75.7th percentile) ===
  weekday     threshold = 0.7440  (score distribution: mean=0.479, std=0.268, p75=0.737)
  weekend     threshold = 0.5631  (score distribution: mean=0.381, std=0.243, p75=0.552)

=== BEFORE vs AFTER CORRECTION ===
         original_flag_rate  corrected_flag_rate  n_rows  absolute_change
regime                                                                   
weekday               0.282                0.243   10164           -0.039
weekend               0.156                0.243    4620            0.087

=== FLAG CHANGE SUMMARY ===
  Total flags changed    : 1,346  (9.1% of all rows)


In [ ]:
RAW_PATH = Path("../data/merged_sensor_metadata.csv")

LOAD_COLS = ["timestamp_utc", "ateccid", "combined_occupancy", "capacity_people", "location_type"]
raw = pd.read_csv(RAW_PATH, usecols=LOAD_COLS)

# ── Fix: ISO8601 handles mixed microsecond/non-microsecond formats ─
raw["timestamp_utc"] = pd.to_datetime(raw["timestamp_utc"], format="ISO8601", utc=True)
print(f"Shape: {raw.shape}")
print(f"Timestamp dtype: {raw['timestamp_utc'].dtype}")

# ── Hour-of-day occupancy breakdown ───────────────────────────
raw["hour"] = raw["timestamp_utc"].dt.hour
print("\n=== Mean combined_occupancy by hour-of-day ===")
print(raw.groupby("hour")["combined_occupancy"].mean().round(3).to_string())

# ── Occupancy ratio ───────────────────────────────────────────
raw["capacity_people"] = raw["capacity_people"].replace(0, np.nan)
raw["occ_ratio"] = raw["combined_occupancy"] / raw["capacity_people"]

# ── Aggregate to daily window per sensor ──────────────────────
raw["window_start"] = raw["timestamp_utc"].dt.normalize()

daily_occ = (
    raw.groupby(["ateccid", "window_start"])
    .agg(
        avg_combined_occ = ("combined_occupancy", "mean"),
        max_combined_occ = ("combined_occupancy", "max"),
        avg_occ_ratio    = ("occ_ratio", "mean"),
        pct_occupied     = ("combined_occupancy", lambda x: (x > 0).mean()),
        capacity_people  = ("capacity_people", "first"),
        location_type    = ("location_type", "first"),
        n_readings       = ("combined_occupancy", "count"),
    )
    .reset_index()
)

print("\n=== DAILY OCCUPANCY TABLE ===")
print(f"Shape: {daily_occ.shape}")
print(daily_occ.head(10).to_string())

print("\n=== avg_occ_ratio distribution ===")
print(daily_occ["avg_occ_ratio"].describe().round(3))

print("\n=== pct_occupied by location_type ===")
print(
    daily_occ.groupby("location_type")["pct_occupied"]
    .mean().round(3).sort_values(ascending=False)
)

# ── Save ──────────────────────────────────────────────────────
daily_occ.to_csv("../data/daily_occupancy.csv", index=False)
print("\nSaved → ../data/daily_occupancy.csv")

Shape: (1048575, 5)
Timestamp dtype: datetime64[us, UTC]

=== Mean combined_occupancy by hour-of-day ===
hour
0     0.000
1     0.000
2     0.000
3     0.000
4     0.000
5     0.010
6     0.096
7     0.497
8     0.719
9     0.777
10    0.803
11    0.678
12    0.698
13    0.706
14    0.633
15    0.249
16    0.057
17    0.024
18    0.016
19    0.005
20    0.001
21    0.002
22    0.000
23    0.000

=== DAILY OCCUPANCY TABLE ===
Shape: (1055, 9)
              ateccid              window_start  avg_combined_occ  max_combined_occ  avg_occ_ratio  pct_occupied  capacity_people location_type  n_readings
0  0123067D065E709F01 2025-08-15 00:00:00+00:00          0.000965                 1       0.000107      0.000965                9  Meeting Room        1036
1  0123067D065E709F01 2025-08-16 00:00:00+00:00          0.000000                 0       0.000000      0.000000                9  Meeting Room         973
2  0123067D065E709F01 2025-08-17 00:00:00+00:00          0.000000                 0   

In [24]:
# ============================================================
# Cell 5b · Occupancy-Adjusted Fault Scoring
#
# Problem:
#   Drift scores are inflated when occupancy is high — sensors
#   naturally diverge when people are present (CO2 rises near
#   occupied desks, noise varies by zone, etc.). This creates
#   occupancy-driven false positives.
#
# Approach — Occupancy Residualisation:
#   1. Join daily avg_occ_ratio onto metric_scores
#   2. For each metric, fit: drift_score ~ occ_ratio (OLS)
#      → β tells us how much score rises per unit occupancy
#   3. Residualise: adj_score = drift_score - β * occ_ratio
#      → removes the occupancy-explained component of each score
#   4. Re-threshold at the same 75.7th percentile on adj_score
#   5. Compare: original → weekday/weekend → occupancy-adjusted
#
# Why residualisation (not stratification)?
#   - Occupancy is continuous (0.0 → 0.26), not binary
#   - Residualisation preserves all windows (no data loss)
#   - β is interpretable and citable
# ============================================================

from scipy import stats

data_dir = Path("data")

# ── Step 1: Load inputs ───────────────────────────────────────
daily_occ = pd.read_csv("../data/daily_occupancy.csv")
daily_occ["window_start"] = pd.to_datetime(daily_occ["window_start"], utc=True)

# Load corrected scores from Cell 5
corrected = pd.read_csv("../data/metric_scores_corrected.csv")
corrected["window_start"] = pd.to_datetime(corrected["window_start"], utc=True)

# ── Step 2: Join occupancy onto scores ────────────────────────
ms = corrected.merge(
    daily_occ[["ateccid", "window_start", "avg_occ_ratio", "pct_occupied",
               "capacity_people", "location_type"]],
    on=["ateccid", "window_start"],
    how="left"
)

# Fill the 1 unmatched window with 0 (no occupancy data = assume empty)
ms["avg_occ_ratio"] = ms["avg_occ_ratio"].fillna(0)
ms["pct_occupied"]  = ms["pct_occupied"].fillna(0)

print(f"Joined shape: {ms.shape}")
print(f"Null occ_ratio: {ms['avg_occ_ratio'].isna().sum()}")

# ── Step 3: Per-metric OLS — drift_score ~ occ_ratio ─────────
print("\n=== PER-METRIC OCCUPANCY COEFFICIENT (β) ===")
print("  β > 0 means occupancy inflates drift score (expected)")
print("  R² tells us how much variance occupancy explains\n")
print(f"  {'metric':<18}  {'β':>8}  {'R²':>8}  {'p-value':>10}  {'interpretation'}")
print(f"  {'-'*17:<18}  {'-'*7:>8}  {'-'*7:>8}  {'-'*9:>10}")

betas = {}
for metric, grp in ms.groupby("metric"):
    x = grp["avg_occ_ratio"].values
    y = grp["metric_driftscore_final"].values
    mask = ~(np.isnan(x) | np.isnan(y))
    if mask.sum() < 10:
        continue
    slope, intercept, r, p, se = stats.linregress(x[mask], y[mask])
    r2 = r**2
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    direction = "↑ inflated by occ" if slope > 0 else "↓ suppressed by occ"
    print(f"  {metric:<18}  {slope:>8.4f}  {r2:>8.4f}  {p:>10.4f}  {sig}  {direction}")
    betas[metric] = slope

# ── Step 4: Residualise scores ────────────────────────────────
ms["beta"] = ms["metric"].map(betas)
ms["occ_explained"] = ms["beta"] * ms["avg_occ_ratio"]
ms["metric_driftscore_adj"] = (ms["metric_driftscore_final"] - ms["occ_explained"]).clip(0, 1)

print(f"\n=== SCORE DISTRIBUTION BEFORE vs AFTER RESIDUALISATION ===")
print(f"  {'':20}  {'original':>10}  {'adjusted':>10}  {'change':>10}")
for stat in ["mean", "std"]:
    orig = getattr(ms["metric_driftscore_final"], stat)()
    adj  = getattr(ms["metric_driftscore_adj"], stat)()
    print(f"  {stat:<20}  {orig:>10.4f}  {adj:>10.4f}  {adj-orig:>+10.4f}")

# ── Step 5: Re-threshold at same 75.7th percentile ───────────
PERCENTILE = 75.7
adj_threshold = np.percentile(ms["metric_driftscore_adj"].dropna(), PERCENTILE)
orig_threshold = np.percentile(ms["metric_driftscore_final"].dropna(), PERCENTILE)

print(f"\n  Original threshold  : {orig_threshold:.4f}")
print(f"  Adjusted threshold  : {adj_threshold:.4f}")

ms["metric_faultflag_occ_adj"] = (ms["metric_driftscore_adj"] >= adj_threshold).astype(int)

# ── Step 6: Three-way comparison ─────────────────────────────
print("\n=== THREE-WAY FLAG RATE COMPARISON ===")
print(f"  {'':20}  {'original':>10}  {'wday/wend':>10}  {'occ_adj':>10}")

# Fleet-wide
o = ms["metric_faultflag_final"].mean()
w = ms["metric_faultflag_corrected"].mean()
a = ms["metric_faultflag_occ_adj"].mean()
print(f"  {'Fleet-wide':<20}  {o:>10.3f}  {w:>10.3f}  {a:>10.3f}")

# By regime
for regime in ["weekday", "weekend"]:
    sub = ms[ms["regime"] == regime]
    o = sub["metric_faultflag_final"].mean()
    w = sub["metric_faultflag_corrected"].mean()
    a = sub["metric_faultflag_occ_adj"].mean()
    print(f"  {regime:<20}  {o:>10.3f}  {w:>10.3f}  {a:>10.3f}")

# By location type
print()
for loc, sub in ms.groupby("location_type"):
    o = sub["metric_faultflag_final"].mean()
    w = sub["metric_faultflag_corrected"].mean()
    a = sub["metric_faultflag_occ_adj"].mean()
    print(f"  {loc:<20}  {o:>10.3f}  {w:>10.3f}  {a:>10.3f}")

# ── Step 7: Day-of-week — does occ_adj flatten it? ───────────
print("\n=== DAY-OF-WEEK FAULT RATE — ALL THREE VERSIONS ===")
ms["day_name"] = ms["window_start"].dt.day_name()
ms["day_num"]  = ms["window_start"].dt.dayofweek
dow = (
    ms.groupby(["day_num","day_name"])
    .agg(
        original   = ("metric_faultflag_final", "mean"),
        wday_wend  = ("metric_faultflag_corrected", "mean"),
        occ_adj    = ("metric_faultflag_occ_adj", "mean"),
    )
    .round(3)
    .reset_index()
    .sort_values("day_num")
)
print(dow[["day_name","original","wday_wend","occ_adj"]].to_string(index=False))

# ── Step 8: Canteen sensors — final picture ───────────────────
print("\n=== CANTEEN SENSORS — THREE-WAY FLAG RATE ===")
CANTEEN = {
    "0123EA796859295001": "S23",
    "0123690E6F25C93201": "S27",
    "01230B43B90B9BB401": "S28",
}
for sid, label in CANTEEN.items():
    sub = ms[ms["ateccid"] == sid]
    o = sub["metric_faultflag_final"].mean()
    w = sub["metric_faultflag_corrected"].mean()
    a = sub["metric_faultflag_occ_adj"].mean()
    print(f"  {label}  original={o:.3f}  wday/wend={w:.3f}  occ_adj={a:.3f}")

# ── Step 9: Per-sensor delta (occ_adj vs original) ───────────
print("\n=== PER-SENSOR: ORIGINAL vs OCC-ADJUSTED FLAG RATE ===")
sensor_3way = (
    ms.groupby("ateccid")
    .agg(
        original = ("metric_faultflag_final", "mean"),
        occ_adj  = ("metric_faultflag_occ_adj", "mean"),
    )
    .round(3)
)
sensor_3way["delta"] = (sensor_3way["occ_adj"] - sensor_3way["original"]).round(3)
sensor_3way = sensor_3way.sort_values("delta")
print("  Biggest REDUCTIONS:")
print(sensor_3way.head(5).to_string())
print("\n  Biggest INCREASES:")
print(sensor_3way.tail(5).to_string())

# ── Step 10: Save ─────────────────────────────────────────────
final = ms[[
    "ateccid", "metric", "window_start", "regime",
    "avg_occ_ratio", "pct_occupied",
    "metric_driftscore_final",
    "metric_driftscore_adj",
    "metric_faultflag_final",
    "metric_faultflag_corrected",
    "metric_faultflag_occ_adj",
    "metric_faulttype_rule",
    "location_type",
]].copy()

final.to_csv("../data/metric_scores_occ_adjusted.csv", index=False)
print("\nSaved → data/metric_scores_occ_adjusted.csv")
print("\nCell 5b complete — ready for Cell 6 (visualisation).")

Joined shape: (14784, 15)
Null occ_ratio: 0

=== PER-METRIC OCCUPANCY COEFFICIENT (β) ===
  β > 0 means occupancy inflates drift score (expected)
  R² tells us how much variance occupancy explains

  metric                     β        R²     p-value  interpretation
  -----------------    -------   -------   ---------
  aqi1                  0.2412    0.0027      0.0927  ns  ↑ inflated by occ
  aqi2                  0.1070    0.0006      0.4430  ns  ↑ inflated by occ
  blue_relative        -0.3412    0.0049      0.0228  *  ↓ suppressed by occ
  clear_relative       -0.6442    0.0166      0.0000  ***  ↓ suppressed by occ
  co2                   0.6383    0.0203      0.0000  ***  ↑ inflated by occ
  green_relative       -0.5587    0.0127      0.0002  ***  ↓ suppressed by occ
  humid                -0.1855    0.0014      0.2265  ns  ↓ suppressed by occ
  lux                  -0.8436    0.0290      0.0000  ***  ↓ suppressed by occ
  pressure             -0.0400    0.0001      0.7762  ns  ↓

In [31]:
# ============================================================
# Cell 6 · Visualisation — Occupancy Correction Analysis
# Uses matplotlib/seaborn — no kaleido required
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

data_dir = Path("../data")
fig_dir  = Path("../figures")
fig_dir.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = sns.color_palette("tab10")

ms = pd.read_csv(data_dir / "metric_scores_occ_adjusted.csv")
ms["window_start"] = pd.to_datetime(ms["window_start"], utc=True)
ms["day_name"]     = ms["window_start"].dt.day_name()
ms["day_num"]      = ms["window_start"].dt.dayofweek

CANTEEN = {
    "0123EA796859295001": "S23",
    "0123690E6F25C93201": "S27",
    "01230B43B90B9BB401": "S28",
}

DAY_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

# ── Figure 1: Day-of-week three-way comparison ────────────────
dow = (
    ms.groupby(["day_num", "day_name"])
    .agg(
        Original     = ("metric_faultflag_final",     "mean"),
        Wkday_Wkend  = ("metric_faultflag_corrected",  "mean"),
        Occ_Adjusted = ("metric_faultflag_occ_adj",    "mean"),
    )
    .round(3).reset_index().sort_values("day_num")
)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(DAY_ORDER))
w = 0.26
bars = [("Original", -w, COLORS[0]),
        ("Wkday_Wkend", 0,  COLORS[1]),
        ("Occ_Adjusted", w,  COLORS[2])]
for col, offset, color in bars:
    ax.bar(x + offset, dow[col], width=w, label=col.replace("_","/"),
           color=color, edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(DAY_ORDER, fontsize=11)
ax.set_xlabel("Day of Week", fontsize=12)
ax.set_ylabel("Mean Fault Rate", fontsize=12)
ax.set_ylim(0, 0.42)
ax.set_title("Fault Rate by Day of Week — Three Scoring Versions\n"
             "Wkday/Wkend correction flattens occupancy bias; Occ-Adj tracks original",
             fontsize=12, pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.01),
          ncol=3, frameon=False, fontsize=10)
ax.axvline(x=4.5, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.text(4.6, 0.38, "weekend →", fontsize=9, color="gray")
plt.tight_layout()
plt.savefig(fig_dir / "fig1_dow_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig1_dow_comparison.png")

# ── Figure 2: Per-metric β coefficient ───────────────────────
beta_data = [
    ("lux",            -0.8436, "***"),
    ("red_relative",   -0.7073, "***"),
    ("clear_relative", -0.6442, "***"),
    ("green_relative", -0.5587, "***"),
    ("temp",           -0.4472, "**"),
    ("blue_relative",  -0.3412, "*"),
    ("humid",          -0.1855, "ns"),
    ("pressure",       -0.0400, "ns"),
    ("voc_resistance",  0.0274, "ns"),
    ("aqi2",            0.1070, "ns"),
    ("tvoc",            0.2107, "ns"),
    ("aqi1",            0.2412, "ns"),
    ("snd",             0.3948, "*"),
    ("co2",             0.6383, "***"),
]
beta_data.sort(key=lambda x: x[1])
labels  = [m for m, b, s in beta_data]
betas   = [b for m, b, s in beta_data]
sigs    = [s for m, b, s in beta_data]
b_colors = ["#e45756" if b > 0 else "#4c78a8" for b in betas]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(labels, betas, color=b_colors, edgecolor="white", height=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=1.2)
for bar, sig, beta in zip(bars, sigs, betas):
    xpos = beta + (0.03 if beta >= 0 else -0.03)
    ha   = "left" if beta >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            sig, va="center", ha=ha, fontsize=10, color="black")
ax.set_xlabel("β coefficient", fontsize=12)
ax.set_title("Occupancy Effect (β) per Metric\n"
             "Red=inflated, Blue=suppressed by occupancy  |  *p<.05  **p<.01  ***p<.001",
             fontsize=11, pad=12)
red_patch  = mpatches.Patch(color="#e45756", label="Inflated by occupancy")
blue_patch = mpatches.Patch(color="#4c78a8", label="Suppressed by occupancy")
ax.legend(handles=[red_patch, blue_patch], loc="lower right", fontsize=10, frameon=False)
ax.set_xlim(-1.1, 0.95)
plt.tight_layout()
plt.savefig(fig_dir / "fig2_beta_metrics.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig2_beta_metrics.png")

# ── Figure 3: Canteen sensors three-way ──────────────────────
canteen_rows = []
for sid, label in CANTEEN.items():
    sub = ms[ms["ateccid"] == sid]
    canteen_rows.append({
        "Sensor":        label,
        "Original":      round(sub["metric_faultflag_final"].mean(), 3),
        "Wkday/Wkend":   round(sub["metric_faultflag_corrected"].mean(), 3),
        "Occ-Adjusted":  round(sub["metric_faultflag_occ_adj"].mean(), 3),
    })
c_df = pd.DataFrame(canteen_rows)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(c_df))
w = 0.26
for i, (col, color) in enumerate([("Original", COLORS[0]),
                                    ("Wkday/Wkend", COLORS[1]),
                                    ("Occ-Adjusted", COLORS[2])]):
    rects = ax.bar(x + (i-1)*w, c_df[col], width=w,
                   label=col, color=color, edgecolor="white")
    for rect in rects:
        ax.text(rect.get_x() + rect.get_width()/2,
                rect.get_height() + 0.015,
                f"{rect.get_height():.2f}",
                ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(c_df["Sensor"], fontsize=12)
ax.set_xlabel("Sensor", fontsize=12)
ax.set_ylabel("Mean Fault Rate", fontsize=12)
ax.set_ylim(0, 0.82)
ax.set_title("Canteen Sensor Fault Rates — Three Scoring Versions\n"
             "All three sensors were markedly under-detected in original scoring",
             fontsize=11, pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.01),
          ncol=3, frameon=False, fontsize=10)
plt.tight_layout()
plt.savefig(fig_dir / "fig3_canteen_threeway.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig3_canteen_threeway.png")

# ── Figure 4: Per-sensor delta ────────────────────────────────
sensor_delta = (
    ms.groupby("ateccid")
    .agg(
        original = ("metric_faultflag_final",  "mean"),
        occ_adj  = ("metric_faultflag_occ_adj", "mean"),
    )
    .reset_index()
)
sensor_delta["delta"] = sensor_delta["occ_adj"] - sensor_delta["original"]
sensor_delta["label"] = sensor_delta["ateccid"].apply(
    lambda x: CANTEEN.get(x, f"...{x[-6:]}")
)
sensor_delta = sensor_delta.sort_values("delta")
d_colors = ["#e45756" if x > 0 else "#4c78a8" for x in sensor_delta["delta"]]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(sensor_delta["label"], sensor_delta["delta"],
        color=d_colors, edgecolor="white", height=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=1.2)
ax.set_xlabel("Δ Fault Rate (Occ-Adj minus Original)", fontsize=12)
ax.set_title("Per-Sensor Fault Rate Change: Occ-Adjusted vs Original\n"
             "Red=more faults detected; Blue=occupancy was inflating faults",
             fontsize=11, pad=12)
red_patch  = mpatches.Patch(color="#e45756", label="More faults detected after adj.")
blue_patch = mpatches.Patch(color="#4c78a8", label="Occupancy was inflating faults")
ax.legend(handles=[red_patch, blue_patch], fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / "fig4_sensor_delta.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig4_sensor_delta.png")

print("\nCell 6 complete — 4 figures saved to figures/")
print("Ready for Cell 7 (Synthetic Fault Injection).")

Saved → figures/fig1_dow_comparison.png
Saved → figures/fig2_beta_metrics.png
Saved → figures/fig3_canteen_threeway.png
Saved → figures/fig4_sensor_delta.png

Cell 6 complete — 4 figures saved to figures/
Ready for Cell 7 (Synthetic Fault Injection).


In [33]:
# ============================================================
# Cell 7 · Synthetic Fault Injection & Detection Evaluation
#
# Purpose:
#   Since no ground truth fault labels exist, we inject known
#   faults into clean sensors and measure whether the pipeline
#   would have caught them. This provides a controlled
#   evaluation of pipeline sensitivity.
#
# Inputs:
#   metric_scores_occ_adjusted.csv  ← corrected baseline
#
# Three injection types (matching the pipeline's taxonomy):
#   1. drift_gradual   — linear offset growing over windows
#   2. spike_transient — single-window impulse
#   3. drift_abrupt    — step change from a specific window
#
# Target sensors:
#   Cleanest non-isolated sensors by occ_adj fault rate,
#   avoiding canteen (S23/27/28) and the S8/S9 pair
#   which are 100% faulted on all windows.
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

data_dir = Path("../data")

# ── Load corrected scores ─────────────────────────────────────
ms = pd.read_csv(data_dir / "metric_scores_occ_adjusted.csv")
ms["window_start"] = pd.to_datetime(ms["window_start"], utc=True)

STUDY_WINDOWS = sorted(ms["window_start"].unique())
N_WINDOWS     = len(STUDY_WINDOWS)
print(f"Study windows: {N_WINDOWS}  ({STUDY_WINDOWS[0]} → {STUDY_WINDOWS[-1]})")

# ── Step 1: Identify cleanest injection targets ───────────────
EXCLUDE = {
    "0123EA796859295001",  # S23 canteen
    "0123690E6F25C93201",  # S27 canteen
    "01230B43B90B9BB401",  # S28 canteen
    "012330E186CB5BA801",  # S8  100% faulted
    "01237028BC6DFD4A01",  # S9  100% faulted
}

sensor_rates = (
    ms[~ms["ateccid"].isin(EXCLUDE)]
    .groupby("ateccid")["metric_faultflag_occ_adj"]
    .mean()
    .sort_values()
)
print("\n=== CLEANEST SENSORS (by occ_adj fault rate) ===")
print(sensor_rates.head(8).round(3).to_string())

# Pick the 3 cleanest non-isolated, non-excluded sensors
TARGET_SENSORS = sensor_rates.head(3).index.tolist()
print(f"\nSelected injection targets: {TARGET_SENSORS}")

# ── Step 2: Injection configuration ──────────────────────────
# Use co2 for gradual/abrupt (clear physical interpretation)
# Use snd for spike (sound spikes are sudden)
# Use temp for abrupt (temperature step changes are common)
INJECTIONS = {
    "drift_gradual": {
        "sensor"     : TARGET_SENSORS[0],
        "metric"     : "co2",
        "start_win"  : STUDY_WINDOWS[4],
        "magnitude"  : 0.25,           # +0.25 added linearly over remaining windows
        "description": "Linear CO2 drift growing from window 5 to end",
    },
    "spike_transient": {
        "sensor"     : TARGET_SENSORS[1],
        "metric"     : "snd",
        "spike_win"  : STUDY_WINDOWS[12],
        "magnitude"  : 0.30,           # single-window +0.30 on sound
        "description": "Single-window sound spike on window 13",
    },
    "drift_abrupt": {
        "sensor"     : TARGET_SENSORS[2],
        "metric"     : "temp",
        "shift_win"  : STUDY_WINDOWS[8],
        "magnitude"  : 0.22,           # step change from window 9 onwards
        "description": "Step temperature change from window 9 onwards",
    },
}

print("\n=== INJECTION CONFIGURATION ===")
for fault_type, cfg in INJECTIONS.items():
    print(f"\n  [{fault_type}]")
    for k, v in cfg.items():
        print(f"    {k}: {v}")

# ── Step 3: Build injected score table ────────────────────────
target_ids = [cfg["sensor"] for cfg in INJECTIONS.values()]
target_metrics = {
    INJECTIONS["drift_gradual"]["sensor"]  : INJECTIONS["drift_gradual"]["metric"],
    INJECTIONS["spike_transient"]["sensor"]: INJECTIONS["spike_transient"]["metric"],
    INJECTIONS["drift_abrupt"]["sensor"]   : INJECTIONS["drift_abrupt"]["metric"],
}

base_rows = []
for sid, metric in target_metrics.items():
    rows = ms[(ms["ateccid"] == sid) & (ms["metric"] == metric)].copy()
    base_rows.append(rows)
base = pd.concat(base_rows).sort_values(["ateccid", "window_start"]).reset_index(drop=True)

# True label column — 0 = clean, 1 = injected fault
base["true_fault"]      = 0
base["injected_score"]  = base["metric_driftscore_adj"].copy()

# Inject drift_gradual
cfg  = INJECTIONS["drift_gradual"]
mask = (base["ateccid"] == cfg["sensor"]) & (base["window_start"] >= cfg["start_win"])
n    = mask.sum()
base.loc[mask, "injected_score"] = (
    base.loc[mask, "metric_driftscore_adj"] + np.linspace(0, cfg["magnitude"], n)
).clip(0, 1)
base.loc[mask, "true_fault"] = 1

# Inject spike_transient
cfg  = INJECTIONS["spike_transient"]
mask = (base["ateccid"] == cfg["sensor"]) & (base["window_start"] == cfg["spike_win"])
base.loc[mask, "injected_score"] = (
    base.loc[mask, "metric_driftscore_adj"] + cfg["magnitude"]
).clip(0, 1)
base.loc[mask, "true_fault"] = 1

# Inject drift_abrupt
cfg  = INJECTIONS["drift_abrupt"]
mask = (base["ateccid"] == cfg["sensor"]) & (base["window_start"] >= cfg["shift_win"])
base.loc[mask, "injected_score"] = (
    base.loc[mask, "metric_driftscore_adj"] + cfg["magnitude"]
).clip(0, 1)
base.loc[mask, "true_fault"] = 1

print(f"\nInjected fault windows : {base['true_fault'].sum()}")
print(f"Clean windows          : {(base['true_fault']==0).sum()}")

# ── Step 4: Apply detection threshold ────────────────────────
# Use the same 75.7th percentile on occ_adj scores
THRESHOLD = np.percentile(ms["metric_driftscore_adj"].dropna(), 75.7)
print(f"\nDetection threshold (75.7th pctile): {THRESHOLD:.4f}")

base["predicted_fault"] = (base["injected_score"] >= THRESHOLD).astype(int)

# ── Step 5: Per-type detection performance ────────────────────
print("\n" + "=" * 60)
print("DETECTION PERFORMANCE BY INJECTION TYPE")
print("=" * 60)

results = []
for fault_type, cfg in INJECTIONS.items():
    sub    = base[base["ateccid"] == cfg["sensor"]]
    y_true = sub["true_fault"]
    y_pred = sub["predicted_fault"]

    tp = int(((y_true==1) & (y_pred==1)).sum())
    fp = int(((y_true==0) & (y_pred==1)).sum())
    fn = int(((y_true==1) & (y_pred==0)).sum())
    tn = int(((y_true==0) & (y_pred==0)).sum())
    p  = precision_score(y_true, y_pred, zero_division=0)
    r  = recall_score(y_true, y_pred, zero_division=0)
    f  = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n  {fault_type}  ({cfg['metric']} on ...{cfg['sensor'][-6:]})")
    print(f"    Injected windows : {y_true.sum()}  ({cfg['description']})")
    print(f"    TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"    Precision : {p:.3f}")
    print(f"    Recall    : {r:.3f}")
    print(f"    F1        : {f:.3f}")

    results.append({"fault_type":fault_type, "metric":cfg["metric"],
                    "TP":tp,"FP":fp,"FN":fn,"TN":tn,
                    "precision":round(p,3),"recall":round(r,3),"f1":round(f,3)})

# ── Step 6: Overall performance ───────────────────────────────
print("\n" + "=" * 60)
print("OVERALL DETECTION PERFORMANCE")
print("=" * 60)
y_all  = base["true_fault"]
yp_all = base["predicted_fault"]
print(f"  Precision : {precision_score(y_all, yp_all, zero_division=0):.3f}")
print(f"  Recall    : {recall_score(y_all, yp_all, zero_division=0):.3f}")
print(f"  F1        : {f1_score(y_all, yp_all, zero_division=0):.3f}")

# ── Step 7: Threshold sensitivity sweep ──────────────────────
print("\n" + "=" * 60)
print("THRESHOLD SENSITIVITY  (effect on F1 across all injection types)")
print("=" * 60)
print(f"  {'Threshold':<12}  {'Precision':<12}  {'Recall':<10}  {'F1':<10}")
print(f"  {'-'*10:<12}  {'-'*9:<12}  {'-'*8:<10}  {'-'*7:<10}")
best_f1, best_thresh = 0, 0
for pct in [60, 65, 70, 75, 75.7, 80, 85, 90]:
    thresh = np.percentile(ms["metric_driftscore_adj"].dropna(), pct)
    preds  = (base["injected_score"] >= thresh).astype(int)
    p = precision_score(y_all, preds, zero_division=0)
    r = recall_score(y_all, preds, zero_division=0)
    f = f1_score(y_all, preds, zero_division=0)
    marker = "  ← current" if pct == 75.7 else ""
    print(f"  {pct:<5.1f} ({thresh:.3f})  {p:<12.3f}  {r:<10.3f}  {f:<10.3f}{marker}")
    if f > best_f1:
        best_f1, best_thresh = f, pct

print(f"\n  Best F1 = {best_f1:.3f} at {best_thresh}th percentile threshold")

# ── Step 8: Injection score timeseries (for visualisation) ───
print("\n" + "=" * 60)
print("INJECTION TIMESERIES SUMMARY (original vs injected score)")
print("=" * 60)
for fault_type, cfg in INJECTIONS.items():
    sub = base[base["ateccid"] == cfg["sensor"]][
        ["window_start","metric_driftscore_adj","injected_score","true_fault","predicted_fault"]
    ]
    n_detected = ((sub["true_fault"]==1) & (sub["predicted_fault"]==1)).sum()
    n_injected = sub["true_fault"].sum()
    print(f"\n  {fault_type}  ({cfg['metric']})  detected {n_detected}/{n_injected} injected windows")
    print(sub.to_string(index=False))

# ── Save ──────────────────────────────────────────────────────
pd.DataFrame(results).to_csv(data_dir / "synthetic_injection_results.csv", index=False)
base.to_csv(data_dir / "synthetic_injection_detail.csv", index=False)
print("\nSaved → data/synthetic_injection_results.csv")
print("\nCell 7 complete — ready for Cell 8 (visualisation of injection).")

Study windows: 32  (2025-08-15 00:00:00+00:00 → 2025-09-15 00:00:00+00:00)

=== CLEANEST SENSORS (by occ_adj fault rate) ===
ateccid
01233545E49BFE5A01    0.011
0123C2ABF23B69DA01    0.016
0123B9C3BE0C085E01    0.022
0123BE9659A3289001    0.025
0123B2DFDFFF08D101    0.036
0123094969F1AA9D01    0.056
0123FCA88283696901    0.058
0123ECAAEA128EAD01    0.058

Selected injection targets: ['01233545E49BFE5A01', '0123C2ABF23B69DA01', '0123B9C3BE0C085E01']

=== INJECTION CONFIGURATION ===

  [drift_gradual]
    sensor: 01233545E49BFE5A01
    metric: co2
    start_win: 2025-08-19 00:00:00+00:00
    magnitude: 0.25
    description: Linear CO2 drift growing from window 5 to end

  [spike_transient]
    sensor: 0123C2ABF23B69DA01
    metric: snd
    spike_win: 2025-08-27 00:00:00+00:00
    magnitude: 0.3
    description: Single-window sound spike on window 13

  [drift_abrupt]
    sensor: 0123B9C3BE0C085E01
    metric: temp
    shift_win: 2025-08-23 00:00:00+00:00
    magnitude: 0.22
    descripti

In [34]:
# ============================================================
# Cell 7b · Threshold Calibration Analysis
#
# Key question: does a sensor-relative threshold fix the
# detection failure revealed in Cell 7?
#
# Three threshold strategies compared:
#   A. Fleet-wide fixed    — current (75.7th pctile of fleet)
#   B. Fleet-wide optimal  — 60th pctile (best F1 from sweep)
#   C. Sensor-relative     — each sensor's own 85th pctile
#                            computed on its pre-injection windows
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

data_dir = Path("../data")

# ── Load injection detail from Cell 7 ────────────────────────
detail = pd.read_csv(data_dir / "synthetic_injection_detail.csv")
detail["window_start"] = pd.to_datetime(detail["window_start"], utc=True)

ms = pd.read_csv(data_dir / "metric_scores_occ_adjusted.csv")
ms["window_start"] = pd.to_datetime(ms["window_start"], utc=True)

y_true = detail["true_fault"]
scores = detail["injected_score"]

# ── Strategy A: Fleet-wide fixed (already done in Cell 7) ────
FLEET_THRESHOLD = np.percentile(ms["metric_driftscore_adj"].dropna(), 75.7)

# ── Strategy B: Fleet-wide optimal (60th pctile from sweep) ──
OPTIMAL_THRESHOLD = np.percentile(ms["metric_driftscore_adj"].dropna(), 60.0)

# ── Strategy C: Sensor-relative threshold ────────────────────
# For each injection sensor, compute the 85th pctile of its
# own CLEAN (pre-injection) windows as the personal threshold
INJECTIONS = {
    "drift_gradual"  : {"sensor": detail["ateccid"].unique()[0],
                        "metric": "co2",   "start_win": "2025-08-19"},
    "spike_transient": {"sensor": detail["ateccid"].unique()[1],
                        "metric": "snd",   "start_win": None},
    "drift_abrupt"   : {"sensor": detail["ateccid"].unique()[2],
                        "metric": "temp",  "start_win": "2025-08-23"},
}

sensor_thresholds = {}
for fault_type, cfg in INJECTIONS.items():
    sid    = cfg["sensor"]
    metric = cfg["metric"]
    # Use all windows BEFORE injection start as the personal baseline
    sensor_hist = ms[(ms["ateccid"] == sid) & (ms["metric"] == metric)]
    if cfg["start_win"]:
        clean = sensor_hist[
            sensor_hist["window_start"] < pd.Timestamp(cfg["start_win"], tz="UTC")
        ]["metric_driftscore_adj"]
    else:
        # spike: all windows except the spike window are "clean"
        clean = sensor_hist["metric_driftscore_adj"]
    pctile_85 = np.percentile(clean.dropna(), 85)
    sensor_thresholds[sid] = pctile_85
    print(f"  {fault_type:<18}  sensor={sid[-6:]}  "
          f"n_clean_windows={len(clean)}  "
          f"85th_pctile={pctile_85:.4f}")

# Build sensor-relative predictions
sensor_rel_preds = detail.apply(
    lambda row: int(row["injected_score"] >= sensor_thresholds.get(row["ateccid"], FLEET_THRESHOLD)),
    axis=1
)

# ── Compare all three strategies ─────────────────────────────
print("\n" + "=" * 65)
print("THRESHOLD STRATEGY COMPARISON")
print("=" * 65)
print(f"\n  {'Strategy':<28}  {'Threshold':<12}  {'P':>6}  {'R':>6}  {'F1':>6}")
print(f"  {'-'*27:<28}  {'-'*10:<12}  {'-'*5:>6}  {'-'*5:>6}  {'-'*5:>6}")

strategies = [
    ("A: Fleet fixed (75.7th pct)",  FLEET_THRESHOLD,   (scores >= FLEET_THRESHOLD).astype(int)),
    ("B: Fleet optimal (60th pct)",  OPTIMAL_THRESHOLD, (scores >= OPTIMAL_THRESHOLD).astype(int)),
    ("C: Sensor-relative (85th pct)", "varies",          sensor_rel_preds),
]
for name, thresh, preds in strategies:
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f = f1_score(y_true, preds, zero_division=0)
    t = f"{thresh:.4f}" if isinstance(thresh, float) else thresh
    print(f"  {name:<28}  {t:<12}  {p:>6.3f}  {r:>6.3f}  {f:>6.3f}")

# ── Per-type breakdown for Strategy C ────────────────────────
print("\n" + "=" * 65)
print("STRATEGY C — PER FAULT TYPE BREAKDOWN")
print("=" * 65)
for fault_type, cfg in INJECTIONS.items():
    sub    = detail[detail["ateccid"] == cfg["sensor"]]
    thresh = sensor_thresholds[cfg["sensor"]]
    preds  = (sub["injected_score"] >= thresh).astype(int)
    yt     = sub["true_fault"]
    tp = int(((yt==1) & (preds==1)).sum())
    fp = int(((yt==0) & (preds==1)).sum())
    fn = int(((yt==1) & (preds==0)).sum())
    p  = precision_score(yt, preds, zero_division=0)
    r  = recall_score(yt, preds, zero_division=0)
    f  = f1_score(yt, preds, zero_division=0)
    print(f"\n  {fault_type}  (threshold={thresh:.4f})")
    print(f"    TP={tp}  FP={fp}  FN={fn}")
    print(f"    Precision={p:.3f}  Recall={r:.3f}  F1={f:.3f}")

# ── Diagnosis: how large would injection need to be ──────────
print("\n" + "=" * 65)
print("REQUIRED MAGNITUDE TO DETECT — FLEET THRESHOLD vs SENSOR THRESHOLD")
print("(how much must be added to the median baseline score to reach threshold)")
print("=" * 65)
for fault_type, cfg in INJECTIONS.items():
    sid    = cfg["sensor"]
    metric = cfg["metric"]
    sensor_hist = ms[(ms["ateccid"] == sid) & (ms["metric"] == metric)]
    median_baseline = sensor_hist["metric_driftscore_adj"].median()
    gap_fleet   = FLEET_THRESHOLD   - median_baseline
    gap_sensor  = sensor_thresholds[sid] - median_baseline
    print(f"\n  {fault_type}  ({metric} on ...{sid[-6:]})")
    print(f"    Median baseline score  : {median_baseline:.4f}")
    print(f"    Fleet threshold (0.706): gap = +{gap_fleet:.4f}  required")
    print(f"    Sensor threshold       : gap = +{gap_sensor:.4f}  required")

# ── Save ──────────────────────────────────────────────────────
summary = []
for name, thresh, preds in strategies:
    summary.append({
        "strategy"  : name,
        "threshold" : thresh,
        "precision" : round(precision_score(y_true, preds, zero_division=0), 3),
        "recall"    : round(recall_score(y_true, preds, zero_division=0), 3),
        "f1"        : round(f1_score(y_true, preds, zero_division=0), 3),
    })
pd.DataFrame(summary).to_csv(data_dir / "threshold_strategy_comparison.csv", index=False)
print("\nSaved → data/threshold_strategy_comparison.csv")
print("\nCell 7b complete — ready for Cell 8 (visualisation).")

  drift_gradual       sensor=FE5A01  n_clean_windows=4  85th_pctile=0.7927
  spike_transient     sensor=085E01  n_clean_windows=32  85th_pctile=0.2058
  drift_abrupt        sensor=69DA01  n_clean_windows=8  85th_pctile=0.2865

THRESHOLD STRATEGY COMPARISON

  Strategy                      Threshold          P       R      F1
  ---------------------------   ----------     -----   -----   -----
  A: Fleet fixed (75.7th pct)   0.7060         0.667   0.038   0.071
  B: Fleet optimal (60th pct)   0.4829         0.850   0.321   0.466
  C: Sensor-relative (85th pct)  varies         0.839   0.491   0.619

STRATEGY C — PER FAULT TYPE BREAKDOWN

  drift_gradual  (threshold=0.7927)
    TP=1  FP=1  FN=27
    Precision=0.500  Recall=0.036  F1=0.067

  spike_transient  (threshold=0.2058)
    TP=24  FP=3  FN=0
    Precision=0.889  Recall=1.000  F1=0.941

  drift_abrupt  (threshold=0.2865)
    TP=1  FP=1  FN=0
    Precision=0.500  Recall=1.000  F1=0.667

REQUIRED MAGNITUDE TO DETECT — FLEET THRESHOLD 

In [37]:
# ============================================================
# Cell 8 · Visualisation — Synthetic Fault Injection Results
#
# Figures produced:
#   5. Threshold strategy comparison (bar — P/R/F1 per strategy)
#   6. Required magnitude gap (fleet vs sensor-relative)
#   7. Injection score timeseries — drift_abrupt (best result)
#   8. Injection score timeseries — drift_gradual (failure case)
# ============================================================

data_dir = Path("../data")
fig_dir  = Path("../figures")
fig_dir.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = sns.color_palette("tab10")

# ── Load data ─────────────────────────────────────────────────
detail   = pd.read_csv(data_dir / "synthetic_injection_detail.csv")
detail["window_start"] = pd.to_datetime(detail["window_start"], utc=True)
detail["window_label"] = detail["window_start"].dt.strftime("%d %b")

ms = pd.read_csv(data_dir / "metric_scores_occ_adjusted.csv")
ms["window_start"] = pd.to_datetime(ms["window_start"], utc=True)

# Corrected sensor assignments (alphabetical sort fix)
SENSOR_MAP = {
    "01233545E49BFE5A01": ("drift_gradual",   "co2",  "2025-08-19"),
    "0123B9C3BE0C085E01": ("drift_abrupt",    "temp", "2025-08-23"),
    "0123C2ABF23B69DA01": ("spike_transient", "snd",  None),
}

# Recompute sensor-relative thresholds with correct assignments
sensor_thresholds = {}
for sid, (fault_type, metric, start_win) in SENSOR_MAP.items():
    hist = ms[(ms["ateccid"] == sid) & (ms["metric"] == metric)]
    if start_win:
        clean = hist[hist["window_start"] < pd.Timestamp(start_win, tz="UTC")]["metric_driftscore_adj"]
    else:
        clean = hist["metric_driftscore_adj"]
    sensor_thresholds[sid] = np.percentile(clean.dropna(), 85)

FLEET_THRESHOLD   = np.percentile(ms["metric_driftscore_adj"].dropna(), 75.7)
OPTIMAL_THRESHOLD = np.percentile(ms["metric_driftscore_adj"].dropna(), 60.0)

# ── Figure 5: Strategy comparison bar chart ───────────────────
strategies = {
    "A: Fleet\nfixed":    {"P": 0.667, "R": 0.038, "F1": 0.071},
    "B: Fleet\noptimal":  {"P": 0.850, "R": 0.321, "F1": 0.466},
    "C: Sensor-\nrelative":{"P": 0.839, "R": 0.491, "F1": 0.619},
}
metrics    = ["Precision", "Recall", "F1"]
x          = np.arange(len(strategies))
labels     = list(strategies.keys())
width      = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, (met, col) in enumerate(zip(["P","R","F1"], COLORS)):
    vals = [strategies[s][met] for s in labels]
    bars = ax.bar(x + (i-1)*width, vals, width=width,
                  label=metrics[i], color=col, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.015,
                f"{bar.get_height():.2f}",
                ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score", fontsize=12)
ax.set_xlabel("Threshold Strategy", fontsize=12)
ax.set_title("Detection Performance by Threshold Strategy\n"
             "Sensor-relative threshold outperforms fleet-wide on clean sensors",
             fontsize=11, pad=12)
ax.legend(loc="upper left", fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / "fig5_strategy_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig5_strategy_comparison.png")

# ── Figure 6: Required magnitude gap ─────────────────────────
fault_labels = ["drift_gradual\n(co2)", "spike_transient\n(snd)", "drift_abrupt\n(temp)"]
sids         = list(SENSOR_MAP.keys())
medians      = []
gaps_fleet   = []
gaps_sensor  = []

for sid, (fault_type, metric, start_win) in SENSOR_MAP.items():
    hist   = ms[(ms["ateccid"] == sid) & (ms["metric"] == metric)]
    median = hist["metric_driftscore_adj"].median()
    medians.append(median)
    gaps_fleet.append(FLEET_THRESHOLD - median)
    gaps_sensor.append(sensor_thresholds[sid] - median)

x   = np.arange(len(fault_labels))
w   = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, gaps_fleet,  width=w, label="Fleet threshold gap",
            color=COLORS[3], edgecolor="white")
b2 = ax.bar(x + w/2, gaps_sensor, width=w, label="Sensor-relative gap",
            color=COLORS[2], edgecolor="white")
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"+{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(fault_labels, fontsize=11)
ax.set_ylabel("Required score increase to trigger flag", fontsize=11)
ax.set_title("Required Fault Magnitude: Fleet vs Sensor-Relative Threshold\n"
             "Sensor-relative threshold is up to 11× more sensitive",
             fontsize=11, pad=12)
ax.legend(fontsize=10, frameon=False)
plt.tight_layout()
plt.savefig(fig_dir / "fig6_magnitude_gap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig6_magnitude_gap.png")

# ── Figure 7: Injection timeseries — drift_abrupt (success) ──
SID_ABRUPT = "0123B9C3BE0C085E01"
THRESH_ABRUPT = sensor_thresholds[SID_ABRUPT]
sub = detail[detail["ateccid"] == SID_ABRUPT].sort_values("window_start")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(sub["window_label"], sub["metric_driftscore_adj"],
        marker="o", color=COLORS[0], label="Original score", linewidth=2)
ax.plot(sub["window_label"], sub["injected_score"],
        marker="s", color=COLORS[1], label="Injected score", linewidth=2,
        linestyle="--")
ax.axhline(FLEET_THRESHOLD,  color=COLORS[3], linestyle=":", linewidth=1.5,
           label=f"Fleet threshold ({FLEET_THRESHOLD:.3f})")
ax.axhline(THRESH_ABRUPT,    color=COLORS[2], linestyle="-.", linewidth=1.5,
           label=f"Sensor threshold ({THRESH_ABRUPT:.3f})")
# Shade injection region
inject_start = sub[sub["true_fault"]==1]["window_label"].iloc[0]
inject_idx   = list(sub["window_label"]).index(inject_start)
ax.axvspan(inject_idx - 0.5, len(sub) - 0.5,
           alpha=0.08, color="red", label="Injection active")
ax.set_xticks(range(len(sub)))
ax.set_xticklabels(sub["window_label"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Drift Score", fontsize=12)
ax.set_title("drift_abrupt Injection — Temp (sensor 085E01)\n"
             "Sensor-relative threshold detects all 24 injected windows (F1=0.941)",
             fontsize=11, pad=12)
ax.legend(fontsize=9, frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(fig_dir / "fig7_injection_abrupt.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig7_injection_abrupt.png")

# ── Figure 8: Injection timeseries — drift_gradual (failure) ──
SID_GRAD   = "01233545E49BFE5A01"
THRESH_GRAD = sensor_thresholds[SID_GRAD]
sub = detail[detail["ateccid"] == SID_GRAD].sort_values("window_start")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(sub["window_label"], sub["metric_driftscore_adj"],
        marker="o", color=COLORS[0], label="Original score", linewidth=2)
ax.plot(sub["window_label"], sub["injected_score"],
        marker="s", color=COLORS[1], label="Injected score", linewidth=2,
        linestyle="--")
ax.axhline(FLEET_THRESHOLD, color=COLORS[3], linestyle=":", linewidth=1.5,
           label=f"Fleet threshold ({FLEET_THRESHOLD:.3f})")
ax.axhline(THRESH_GRAD,     color=COLORS[2], linestyle="-.", linewidth=1.5,
           label=f"Sensor threshold ({THRESH_GRAD:.3f}) — inflated by Aug 17 spike")
inject_start = sub[sub["true_fault"]==1]["window_label"].iloc[0]
inject_idx   = list(sub["window_label"]).index(inject_start)
ax.axvspan(inject_idx - 0.5, len(sub) - 0.5,
           alpha=0.08, color="red", label="Injection active")
ax.set_xticks(range(len(sub)))
ax.set_xticklabels(sub["window_label"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Drift Score", fontsize=12)
ax.set_title("drift_gradual Injection — CO2 (sensor FE5A01)\n"
             "Sensor threshold inflated by pre-injection spike — calibration history too short",
             fontsize=11, pad=12)
ax.legend(fontsize=9, frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig(fig_dir / "fig8_injection_gradual.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig8_injection_gradual.png")

print("\nCell 8 complete — 4 figures saved.")
print("Ready for Cell 9 (S8/S9 case study) or dissertation write-up.")

Saved → figures/fig5_strategy_comparison.png
Saved → figures/fig6_magnitude_gap.png
Saved → figures/fig7_injection_abrupt.png
Saved → figures/fig8_injection_gradual.png

Cell 8 complete — 4 figures saved.
Ready for Cell 9 (S8/S9 case study) or dissertation write-up.


In [38]:
# ============================================================
# Cell 9 · Case Study — Sensors S8 & S9 (Open Space)
#
# Why these sensors:
#   S8 (012330E186CB5BA801) and S9 (01237028BC6DFD4A01) have
#   mean fault run lengths of 9.57 and 9.41 — the highest in
#   the fleet — and fault rates of 1.0 on temp, humid, snd,
#   lux and all light channels across ALL 32 windows.
#
# Questions answered:
#   1. Do S8/S9 fault together (synchronised = shared cause)?
#   2. Are their neighbours (S7, S10, S14, S15, S16) clean?
#   3. Which metrics drive the faults — all or specific?
#   4. Is the fault pattern consistent or episodic?
#   5. Does the occupancy-adjusted score change the picture?
# ============================================================

sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = sns.color_palette("tab10")

# ── Load data ─────────────────────────────────────────────────
ms = pd.read_csv(data_dir / "metric_scores_occ_adjusted.csv")
ms["window_start"] = pd.to_datetime(ms["window_start"], utc=True)
ms["window_label"] = ms["window_start"].dt.strftime("%d %b")
ms["day_num"]      = ms["window_start"].dt.dayofweek

# ── Sensor definitions ────────────────────────────────────────
S8  = "012330E186CB5BA801"
S9  = "01237028BC6DFD4A01"
S7  = "01235C86E052AE3F01"
S10 = "0123BE9659A3289001"
S14 = "0123B2DFDFFF08D101"
S15 = "01233545E49BFE5A01"
S16 = "0123DD0A4D7EBD8801"

GROUP = {S8:"S8", S9:"S9", S7:"S7", S10:"S10", S14:"S14", S15:"S15", S16:"S16"}
FOCAL = [S8, S9]
NEIGHBOURS = [S7, S10, S14, S15, S16]

# ── Q1: Full metric fault profile ─────────────────────────────
print("=== FAULT RATE BY METRIC (original vs occ_adj) ===")
metric_profile = (
    ms[ms["ateccid"].isin(GROUP.keys())]
    .groupby(["ateccid", "metric"])
    .agg(
        original = ("metric_faultflag_final",    "mean"),
        occ_adj  = ("metric_faultflag_occ_adj",  "mean"),
    )
    .round(3)
    .reset_index()
)
metric_profile["label"] = metric_profile["ateccid"].map(GROUP)
pivot_orig = metric_profile.pivot(index="label", columns="metric", values="original")
pivot_adj  = metric_profile.pivot(index="label", columns="metric", values="occ_adj")

print("\nOriginal fault rates:")
print(pivot_orig.to_string())
print("\nOcc-adjusted fault rates:")
print(pivot_adj.to_string())

# ── Q2: Synchronisation — do S8 and S9 fault together? ───────
print("\n=== S8 / S9 FAULT SYNCHRONISATION ===")
print("(how often do both flag on the same window for the same metric)")
sync_results = []
for metric in ms["metric"].unique():
    sub = ms[
        (ms["ateccid"].isin([S8, S9])) &
        (ms["metric"] == metric)
    ].pivot(index="window_start", columns="ateccid", values="metric_faultflag_occ_adj")
    if S8 not in sub.columns or S9 not in sub.columns:
        continue
    both  = int(((sub[S8]==1) & (sub[S9]==1)).sum())
    s8_only = int(((sub[S8]==1) & (sub[S9]==0)).sum())
    s9_only = int(((sub[S8]==0) & (sub[S9]==1)).sum())
    neither = int(((sub[S8]==0) & (sub[S9]==0)).sum())
    sync_pct = both / max(sub[S8].sum(), 1) * 100
    sync_results.append({
        "metric": metric, "both": both, "S8_only": s8_only,
        "S9_only": s9_only, "neither": neither, "sync_%": round(sync_pct, 1)
    })
    print(f"  {metric:<18}  both={both:>2}  S8_only={s8_only}  "
          f"S9_only={s9_only}  neither={neither:>2}  sync={sync_pct:.0f}%")

# ── Q3: Neighbour comparison on critical metrics ──────────────
CRITICAL = ["temp", "humid", "snd", "lux", "co2"]
print("\n=== CRITICAL METRIC DRIFT SCORES: S8/S9 vs NEIGHBOURS ===")
for metric in CRITICAL:
    sub = ms[
        (ms["ateccid"].isin(GROUP.keys())) &
        (ms["metric"] == metric)
    ].groupby("ateccid")["metric_driftscore_adj"].agg(["mean","std"]).round(3)
    sub.index = sub.index.map(GROUP)
    print(f"\n  {metric}:")
    print(sub.to_string())

# ── Q4: Temporal pattern — are faults present across all weeks?
print("\n=== S8/S9 WEEKLY FAULT RATE (occ_adj) ===")
ms["week"] = ms["window_start"].dt.isocalendar().week
for sid, label in [(S8,"S8"), (S9,"S9")]:
    weekly = (
        ms[(ms["ateccid"]==sid)]
        .groupby("week")["metric_faultflag_occ_adj"]
        .mean().round(3)
    )
    print(f"\n  {label}: {weekly.to_dict()}")

# ── Q5: Metric-level drift score distributions ───────────────
print("\n=== S8 vs FLEET MEDIAN DRIFT SCORE (per metric) ===")
fleet_med  = ms.groupby("metric")["metric_driftscore_adj"].median().round(3)
s8_med     = ms[ms["ateccid"]==S8].groupby("metric")["metric_driftscore_adj"].median().round(3)
s9_med     = ms[ms["ateccid"]==S9].groupby("metric")["metric_driftscore_adj"].median().round(3)
comparison = pd.DataFrame({"fleet_median": fleet_med, "S8_median": s8_med, "S9_median": s9_med})
comparison["S8_vs_fleet"] = (comparison["S8_median"] - comparison["fleet_median"]).round(3)
comparison["S9_vs_fleet"] = (comparison["S9_median"] - comparison["fleet_median"]).round(3)
print(comparison.sort_values("S8_vs_fleet", ascending=False).to_string())

# ── Save diagnostic table ──────────────────────────────────────
comparison.to_csv(data_dir / "s8s9_case_study.csv")
print("\nSaved → data/s8s9_case_study.csv")

# ================================================================
# Figure 9: Heatmap — fault flag (occ_adj) per sensor per metric
# ================================================================
heatmap_data = (
    ms[ms["ateccid"].isin(GROUP.keys())]
    .groupby(["ateccid", "metric"])["metric_faultflag_occ_adj"]
    .mean()
    .reset_index()
)
heatmap_data["label"] = heatmap_data["ateccid"].map(GROUP)
heat_pivot = heatmap_data.pivot(index="label", columns="metric", values="metric_faultflag_occ_adj")
# Order rows: S8/S9 first, then neighbours
row_order = ["S8","S9","S7","S16","S15","S14","S10"]
heat_pivot = heat_pivot.reindex([r for r in row_order if r in heat_pivot.index])

fig, ax = plt.subplots(figsize=(14, 4.5))
sns.heatmap(
    heat_pivot, ax=ax, annot=True, fmt=".2f", cmap="Reds",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Fault Rate", "shrink": 0.8},
    vmin=0, vmax=1,
)
ax.set_title("Fault Rate Heatmap (Occ-Adjusted) — S8/S9 and Spatial Neighbours\n"
             "S8 and S9 fault at 1.0 on temp, humid, snd, lux — neighbours remain clean",
             fontsize=11, pad=12)
ax.set_xlabel("Metric", fontsize=11)
ax.set_ylabel("Sensor", fontsize=11)
ax.axhline(2, color="navy", linewidth=2, linestyle="--")
ax.text(heat_pivot.shape[1]+0.05, 1, "← S8/S9", va="center", fontsize=9, color="navy")
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(fig_dir / "fig9_s8s9_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig9_s8s9_heatmap.png")

# ================================================================
# Figure 10: Timeseries — S8/S9 vs S7/S10 on temp and snd
# ================================================================
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
WINDOWS = sorted(ms["window_start"].unique())
labels  = [pd.Timestamp(w).strftime("%d %b") for w in WINDOWS]

for ax, metric, title in zip(
    axes,
    ["temp", "snd"],
    ["Temperature drift score — S8/S9 vs neighbours",
     "Sound drift score — S8/S9 vs neighbours"]
):
    for sid, label, color, lw, ls in [
        (S8,  "S8",  COLORS[3], 2.5, "-"),
        (S9,  "S9",  COLORS[0], 2.5, "--"),
        (S7,  "S7",  COLORS[2], 1.2, "-"),
        (S10, "S10", COLORS[4], 1.2, "-"),
        (S16, "S16", COLORS[5], 1.2, "-"),
    ]:
        sub = ms[(ms["ateccid"]==sid) & (ms["metric"]==metric)].sort_values("window_start")
        if sub.empty:
            continue
        ax.plot(range(len(sub)), sub["metric_driftscore_adj"],
                label=label, color=color, linewidth=lw, linestyle=ls, marker="o", markersize=3)
    fleet_thresh = np.percentile(ms["metric_driftscore_adj"].dropna(), 75.7)
    ax.axhline(fleet_thresh, color="gray", linestyle=":", linewidth=1,
               label=f"Fleet threshold ({fleet_thresh:.3f})")
    ax.set_ylabel("Drift Score", fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9, frameon=False, loc="upper right", ncol=6)
    ax.set_ylim(0, 1.05)

axes[-1].set_xticks(range(len(labels)))
axes[-1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
axes[-1].set_xlabel("Window", fontsize=11)
fig.suptitle("S8/S9 vs Spatial Neighbours — Temp & Sound Drift Scores\n"
             "S8/S9 consistently above threshold; neighbours remain below",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(fig_dir / "fig10_s8s9_timeseries.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → figures/fig10_s8s9_timeseries.png")

print("\nCell 9 complete — ready for dissertation write-up structure.")

=== FAULT RATE BY METRIC (original vs occ_adj) ===

Original fault rates:
metric   aqi1   aqi2  blue_relative  clear_relative    co2  green_relative  humid    lux  pressure  red_relative  snd  temp   tvoc  voc_resistance
label                                                                                                                                             
S10     0.219  0.250          0.000           0.000  0.000           0.000    0.0  0.000     0.000         0.000  0.0   0.0  0.000           0.000
S14     0.094  0.094          0.000           0.000  0.000           0.000    0.0  0.000     0.000         0.000  0.0   0.0  0.000           0.000
S15     0.031  0.031          0.312           0.000  0.281           0.000    0.0  0.000     0.000         0.031  0.0   0.0  0.031           0.000
S16     0.031  0.000          0.031           0.188  0.000           0.656    0.0  0.719     0.031         0.062  0.0   0.0  0.000           0.188
S7      0.000  0.031          0.000         